In [ ]:
# Standard library
import os
import re
import glob
import warnings
from pathlib import Path
from collections import Counter, defaultdict

# Third-party scientific stack
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Local project utilities
from utils.exp_validation import (
    parse_mixcr,
    extract_metrics_from_report,
    extract_metrics_from_assemble,
    get_expansion,
    plot_violin_outline,
    plot_bars_twin,
    Upset_public_tcrs,
    summarize_overlap_conv,
    get_overlap_dataframe,
    plot_overlap_nested,
    plot_MAIT_per_patient,
    compute_overlap_simple,
    compute_stats,
    compute_overlap_conv_with_stats,
    compute_overlap_clusters_conv,
    plot_relative_overlap,
    build_overview,
)

# Global settings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

os.environ["OMP_NUM_THREADS"] = "8"

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "Arial"

In [ ]:
# Base directory of the files
# base_dir = Path("..").resolve()
base_dir = Path(
    "/Users/romivandoren/Desktop/Romi/PhD/TCR_Microbiome/0_AIRRWAS_manuscript/X_git_repo"
).resolve()

# Directories containing the processed files
raw_exp1_data = base_dir / "1_raw_data" / "3_Experimental_data" / "1_PB_PL"
raw_exp2_data = base_dir / "1_raw_data" / "3_Experimental_data" / "2_PL"

exp_processed_data = base_dir / "2_processed_data" / "10_experimental_validation"
discovery_processed = base_dir / "2_processed_data" / "0_discovery_data"
discovery_conv = base_dir / "2_processed_data" / "6_combined_convergence"

# Output directories for the figures and tables
figures_dir = base_dir / "3_figures" / "Figures"
tables_dir = base_dir / "3_figures" / "Tables"

# Experiment 1: PBMC stimulation with Prevotella bivia and Peptoniphilus lacrimalis lysates

## 1. Get Mixcr data, parse and get counts

In [ ]:
# Directory containing your fastq.gz files from the stimulation
sampledir = raw_exp1_data / "1_raw_fastq_pat"

r1_files = {}
r2_files = {}


# Loop through the files
for filename in os.listdir(sampledir):
    if filename.endswith("R1_001.fastq.gz"):
        patient_name = filename.split("_R")[0]
        r1_files[patient_name] = os.path.join(sampledir, filename)
    elif filename.endswith("R2_001.fastq.gz"):
        patient_name = filename.split("_R")[0]
        r2_files[patient_name] = os.path.join(sampledir, filename)

# Find common patient names and create file pairs
common_patients = set(r1_files.keys()).intersection(r2_files.keys())
file_pairs = [(r1_files[patient], r2_files[patient]) for patient in common_patients]


# Extract metrics from report files and create a DataFrame
resultdir = raw_exp1_data / "2_Mixcr_results"
metrics_list = []
for patient in common_patients:
    report_file = os.path.join(resultdir, f"{patient}.align.report.txt")
    if os.path.exists(report_file):
        metrics = extract_metrics_from_report(report_file)
        metrics["Patient"] = patient
        metrics_list.append(metrics)
    assemble_file = os.path.join(resultdir, f"{patient}.assemble.report.txt")
    if os.path.exists(assemble_file):
        assemble_metrics = extract_metrics_from_assemble(assemble_file)
        assemble_metrics["Patient"] = patient
        metrics.update(assemble_metrics)

# Create a DataFrame from the metrics list
mixcr_stats = pd.DataFrame(metrics_list)
mixcr_stats["Patient_id"] = mixcr_stats["Patient"].str.split("_").str[0]
mixcr_stats["condition"] = mixcr_stats["Patient"].str.extract(
    r"(PB_1d|PB_1w|PL_1d|PL_1w)", expand=False
)
mixcr_stats["full_id"] = mixcr_stats["Patient_id"] + "_" + mixcr_stats["condition"]
mixcr_stats = mixcr_stats.sort_values(by="Patient")

patient_id_list = set(mixcr_stats["full_id"])


# Define a function to read in the data for each chain type and combine
def process_chain(chain_type):
    pattern = r"^{}.*\.clones.{}\.tsv$"
    dfs = []

    for name in patient_id_list:
        regex_pattern = pattern.format(re.escape(name), chain_type)
        compiled_regex = re.compile(regex_pattern)

        for filename in os.listdir(resultdir):
            if compiled_regex.match(filename):
                file_path = os.path.join(resultdir, filename)
                df = pd.read_csv(file_path, sep="\t")
                df["patient"] = name
                df["run_id"] = filename.split(".")[0]
                df["chain"] = chain_type
                dfs.append(df)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


# Process each chain type
all_chains = ["TRA", "TRB", "TRG", "TRD"]
all_dataframes = [process_chain(chain) for chain in all_chains]

total_df = pd.concat(all_dataframes, ignore_index=True)
total_df["patient_id"] = total_df["patient"].str.split("_").str[0]

total_df["allVHitsWithScore"] = total_df["allVHitsWithScore"].replace(
    r"(.*)DV(\d+)", r"\1/DV\2", regex=True
)

# Parse the TCR data
Full_parsed = parse_mixcr(total_df, rename=True)

Full_parsed["condition"] = (
    Full_parsed["patient"].str.split("_").str[1]
    + "_"
    + Full_parsed["patient"].str.split("_").str[2]
)

final_data = Full_parsed[
    [
        "readCount",
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient",
        "run_id",
        "chain",
        "patient_id",
        "condition",
    ]
]

# Add clone counts to the dataframe
final_data["clonecount"] = final_data.groupby(
    ["junction", "junction_aa", "v_call", "j_call", "patient", "condition"]
)["readCount"].transform("sum")
final_data.drop_duplicates(
    subset=["junction", "junction_aa", "v_call", "j_call", "patient", "condition"],
    inplace=True,
)

# Group by subgroup and get the number of unique TCRs per group
grouped = (
    final_data.groupby(["patient_id", "condition", "chain"])["junction_aa"]
    .nunique()
    .reset_index()
    .pivot_table(
        index=["patient_id", "condition"],
        columns="chain",
        values="junction_aa",
        fill_value=0,
    )
    .reset_index()
)

grouped.columns.name = None
grouped = grouped.rename(columns={"patient_id": "patient"})

## 2. Get clonal expansion per condition and patient

In [ ]:
data_pb_1d = final_data[final_data["condition"] == "PB_1d"]
data_pb_1w = final_data[final_data["condition"] == "PB_1w"]
data_pl_1d = final_data[final_data["condition"] == "PL_1d"]
data_pl_1w = final_data[final_data["condition"] == "PL_1w"]

# Only 3 TCRs found -- remove because gives unrealistic trend
data_pb_1w = data_pb_1w[~data_pb_1w["patient_id"].isin(["PT040", "PT062"])]

data_exp = [
    get_expansion(data_pb_1d, "patient", "patient_id", "frequency"),
    get_expansion(data_pb_1w, "patient", "patient_id", "frequency"),
    get_expansion(data_pl_1d, "patient", "patient_id", "frequency"),
    get_expansion(data_pl_1w, "patient", "patient_id", "frequency"),
]

## 3. Create figures 

### 3A. Violin and barplot for counts

In [ ]:
# Figure parameters and variables
label_map = {"PB_1d": "1 Day", "PB_1w": "1 Week", "PL_1d": "1 Day", "PL_1w": "1 Week"}

chains = ["TRA", "TRB"]
pb_conditions = ["PB_1d", "PB_1w"]
pl_conditions = ["PL_1d", "PL_1w"]

palette_tcr = {
    "PB_1d_TRA": "#003049",
    "PB_1d_TRB": "#669bbc",
    "PB_1w_TRA": "#6C0F32",
    "PB_1w_TRB": "#FF858D",
    "PL_1d_TRA": "#003049",
    "PL_1d_TRB": "#669bbc",
    "PL_1w_TRA": "#6C0F32",
    "PL_1w_TRB": "#FF858D",
}


#########################
### Figure generation ###
#########################

fig = plt.figure(figsize=(20, 8))
gs = fig.add_gridspec(1, 5, width_ratios=[0.5, 1.8, 0.6, 0.5, 1.5], wspace=0.05)
axs = [fig.add_subplot(gs[0, i]) for i in range(5)]

plot_violin_outline(axs[0], pb_conditions, mixcr_stats, label_map)
plot_bars_twin(axs[1], pb_conditions, grouped, label_map, chains, palette_tcr)

plot_violin_outline(axs[3], pl_conditions, mixcr_stats, label_map)
plot_bars_twin(axs[4], pl_conditions, grouped, label_map, chains, palette_tcr)

axs[0].set_title("", fontsize=16)
axs[1].set_title("", fontsize=16)
axs[2].axis("off")
axs[3].set_title("", fontsize=16)
axs[4].set_title("", fontsize=16)

plt.tight_layout()
plt.savefig(figures_dir / "S9A_validation1_counts.pdf", bbox_inches="tight")
plt.show()

### 3B. Expansion plots

In [ ]:
# Figure parameters and variables
colors = ["#feedde", "#fdbe85", "#fd8d3c", "#e6550d", "#a63603"]
fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)

# Combine the data for 1 day and 1 week together
bacterium_data = {
    "PB": [data_exp[0], data_exp[1]],
    "PL": [data_exp[2], data_exp[3]],
}

#########################
### Figure generation ###
#########################

for ax, (bact, dfs) in zip(axes, bacterium_data.items()):
    all_patients = sorted(set(dfs[0].index).union(dfs[1].index))
    dfs_aligned = [df.reindex(all_patients, fill_value=0) for df in dfs]
    df_concat = pd.concat(dfs_aligned, keys=["1 Day", "1 Week"])
    x_labels = [f"{pid}" for day, pid in df_concat.index]

    df_concat.plot(
        kind="bar",
        stacked=True,
        color=colors[: len(df_concat.columns)],
        width=0.9,
        ax=ax,
        legend=False,
    )

    ax.set_xticks(range(len(df_concat)))
    ax.set_xticklabels(x_labels, rotation=90, ha="center", fontsize=12)
    ax.set_title("")
    ax.set_xlabel("", fontsize=14)
    ax.set_ylabel("Frequency", fontsize=14)
    ax.set_ylim(0, 1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(labels),
    fontsize=14,
    frameon=True,
    bbox_to_anchor=(0.5, -0.05),
)

plt.tight_layout()
plt.savefig(figures_dir / "S9B_validation1_expansion.pdf", bbox_inches="tight")
plt.show()

### 3C. Overlap heatmaps and upset plots per bacterium

In [ ]:
# Create separate dataframes for each bacterium
patients_pb = final_data[final_data["condition"].str.startswith("PB")][
    "patient_id"
].unique()
PB_data = final_data[final_data["condition"].str.startswith("PB")]

patients_pl = final_data[final_data["condition"].str.startswith("PL")][
    "patient_id"
].unique()
PL_data = final_data[final_data["condition"].str.startswith("PL")]


# Function to build a matrix describing the overlap in TCRs
def build_overlap_matrix(patients, df):
    matrix = pd.DataFrame(0, index=patients, columns=patients)
    for i in patients:
        clones_i = set(df[df["patient_id"] == i]["junction_aa"])
        for j in patients:
            clones_j = set(df[df["patient_id"] == j]["junction_aa"])
            matrix.loc[i, j] = len(clones_i & clones_j)
    return matrix


# Create overlap matrix for each bacterium
overlap_pb = build_overlap_matrix(patients_pb, PB_data)
overlap_pl = build_overlap_matrix(patients_pl, PL_data)
mask_pb = np.eye(len(overlap_pb), dtype=bool)
mask_pl = np.eye(len(overlap_pl), dtype=bool)


#########################
### Figure generation ###
#########################

### Heatmaps for bacterium-specific overlap between patients ###
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

sns.heatmap(
    overlap_pb,
    annot=True,
    fmt="d",
    cmap="Purples",
    mask=mask_pb,
    cbar_kws={"label": "Shared TCR count"},
    ax=axes[0],
)
axes[0].set_title("PB - Patient TCR overlap")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=90)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)

sns.heatmap(
    overlap_pl,
    annot=True,
    fmt="d",
    cmap="Greens",
    mask=mask_pl,
    cbar_kws={"label": "Shared TCR count"},
    ax=axes[1],
)
axes[1].set_title("PL - Patient TCR overlap")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=90)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(figures_dir / "S9C_D_validation1_heatmap_overlap.pdf", bbox_inches="tight")
plt.show()


### Upset plots for bacterium-specific overlap between patients ###
pb_public = Upset_public_tcrs(
    final_data, figures_dir, condition_prefix="PB", min_shared=2
)
pl_public = Upset_public_tcrs(
    final_data, figures_dir, condition_prefix="PL", min_shared=2
)

## 4. Overlap with original discovery cohort

### 4A. Overlap table per patient and bacterium

In [ ]:
# Original discovery dataset
disc_data = pd.read_csv(discovery_processed / "0_final_data_parsed_annotated.csv")
disc_data = disc_data[
    ["junction_aa", "v_call", "j_call", "patient", "cell_type", "chain", "full_tcr"]
]

# All convergent TCRs
final_conv = pd.read_csv(discovery_conv / "0_combined_final_sign_interactions.csv")
final_conv = final_conv[
    (final_conv["dataset"] == "Discovery")
    & (final_conv["convergence"] > 2)
    & (final_conv["pvalue"] < 0.05)
].rename(columns={"patient_id": "patient"})
final_conv["chain"] = final_conv["v_call"].str.extract(r"(TR[A|B])")

# Experimental stimulation data
final_data["full_tcr"] = (
    final_data["v_call"] + "_" + final_data["junction_aa"] + "_" + final_data["j_call"]
)

# Split the data per bacterium
PL_data = final_data[final_data["condition"].isin(["PL_1d", "PL_1w"])]
PB_data = final_data[final_data["condition"].isin(["PB_1d", "PB_1w"])]

PB_set = set(PB_data["junction_aa"])
PL_set = set(PL_data["junction_aa"])


overlap = PB_set & PL_set
print(len(PB_set), len(PL_set), len(overlap))
overlapping_rows = final_data[final_data["junction_aa"].isin(overlap)].drop_duplicates(
    subset=["junction_aa"]
)
overlapping_rows_MAIT = overlapping_rows[
    (overlapping_rows["v_call"] == "TRAV1-2*01")
    & (
        overlapping_rows["j_call"].isin(
            ["TRAJ33*01", "TRAJ12*01", "TRAJ20*01", "TRAJ30*01"]
        )
    )
]

print(len(overlapping_rows_MAIT))
print(
    len(
        final_data[final_data["junction_aa"].isin(overlapping_rows_MAIT["junction_aa"])]
    )
)

pb_summary = summarize_overlap_conv(PB_data, disc_data, final_conv, "PB")
pl_summary = summarize_overlap_conv(PL_data, disc_data, final_conv, "PL")
data_summary_full = pd.concat([pb_summary, pl_summary], ignore_index=True)
data_summary_full = data_summary_full.set_index(["patient_id", "cell_type"])

data_summary_full.to_csv(
    exp_processed_data / "1_PB_PL" / "2_overlap_overview_numbers.csv"
)


# Find the stimulated TCRs that were convergent in the discovery data for Prevotella
overlap_conv_PB = pd.merge(PB_data, final_conv, on="junction_aa", how="inner")
overlap_conv_PB = overlap_conv_PB[
    [
        "junction_aa",
        "patient_id",
        "condition",
        "clonecount",
        "patient_y",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
    ]
].drop_duplicates()
overlap_conv_PB.to_csv(exp_processed_data / "1_PB_PL" / "2_all_conv_overlap_PB.csv")


# Find the stimulated TCRs that were convergent in the discovery data for Prevotella
overlap_conv_PL = pd.merge(PL_data, final_conv, on="junction_aa", how="inner")
overlap_conv_PL = overlap_conv_PL[
    [
        "junction_aa",
        "patient_id",
        "condition",
        "clonecount",
        "patient_y",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
    ]
].drop_duplicates()
overlap_conv_PL.to_csv(exp_processed_data / "1_PB_PL" / "2_all_conv_overlap_PL.csv")

### 4B. Total counts, overlap with discovery and overlap with convergent TCRs

In [ ]:
# Build PB and PL dataframes directly
pb_plot_sorted, pb_labels = get_overlap_dataframe(PB_data, disc_data, final_conv, "PB")
pl_plot_sorted, pl_labels = get_overlap_dataframe(PL_data, disc_data, final_conv, "PL")

fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"hspace": 0.5})

plot_overlap_nested(
    axes[0],
    pb_plot_sorted,
    pb_labels,
    "Prevotella bivia overlap",
    annotate_percent=False,
)
plot_overlap_nested(
    axes[1],
    pl_plot_sorted,
    pl_labels,
    "Peptoniphilus lacrimalis overlap",
    annotate_percent=False,
)

plt.tight_layout()
fig.savefig(figures_dir / "6A_Overlap_per_bacterium.pdf", bbox_inches="tight")
plt.show()

### 4C. Top expanded clonotypes

In [ ]:
# Ad clone count and clone frequency information to the data per bacterium
PB_data["pat_clone_counts"] = PB_data.groupby(["junction_aa", "patient_id"])[
    "clonecount"
].transform("sum")

PL_data["pat_clone_counts"] = PL_data.groupby(["junction_aa", "patient_id"])[
    "clonecount"
].transform("sum")

pb_totals = PB_data.groupby("patient_id")["clonecount"].transform("sum")
pl_totals = PL_data.groupby("patient_id")["clonecount"].transform("sum")

PB_data["pat_clone_freq"] = PB_data["pat_clone_counts"] / pb_totals
PL_data["pat_clone_freq"] = PL_data["pat_clone_counts"] / pl_totals

PB_nodup = PB_data.drop_duplicates(subset=["junction_aa", "patient_id"])
PL_nodup = PL_data.drop_duplicates(subset=["junction_aa", "patient_id"])


# Determine publicity of TCRs per bacterium group and extract public TCRs
PB_count = PB_nodup.groupby("junction_aa")["patient_id"].nunique()
PB_nodup["public"] = PB_nodup["junction_aa"].map(PB_count)
public_PB = PB_nodup[PB_nodup["public"] > 1]


pl_count = PL_nodup.groupby("junction_aa")["patient_id"].nunique()
PL_nodup["public"] = PL_nodup["junction_aa"].map(pl_count)
public_PL = PL_nodup[PL_nodup["public"] > 1]

# Determine the total shared frequency across patients per public TCRs
public_PB["shared_freq"] = public_PB.groupby(["junction_aa"])[
    "pat_clone_freq"
].transform("sum")

public_PL["shared_freq"] = public_PL.groupby(["junction_aa"])[
    "pat_clone_freq"
].transform("sum")


# Select top 20 most expanded clonotypes
top20_pb = public_PB.groupby("junction_aa")["shared_freq"].mean().nlargest(20).index
top20_pl = public_PL.groupby("junction_aa")["shared_freq"].mean().nlargest(20).index

# Pivot tables (clonotypes × patients)
pb_heatmap = (
    public_PB[public_PB["junction_aa"].isin(top20_pb)]
    .groupby(["junction_aa", "patient_id"])["pat_clone_freq"]
    .sum()
    .unstack(fill_value=0)
)

pl_heatmap = (
    public_PL[public_PL["junction_aa"].isin(top20_pl)]
    .groupby(["junction_aa", "patient_id"])["pat_clone_freq"]
    .sum()
    .unstack(fill_value=0)
)

# Reorder rows of the pivoted heatmap by mean frequency
pb_heatmap = pb_heatmap.loc[pb_heatmap.sum(axis=1).sort_values(ascending=False).index]
pl_heatmap = pl_heatmap.loc[pl_heatmap.sum(axis=1).sort_values(ascending=False).index]

pb_plot = np.sqrt(pb_heatmap)
pl_plot = np.sqrt(pl_heatmap)

vmin = 0
vmax = max(pb_plot.max().max(), pl_plot.max().max())

cmap_prev = sns.color_palette("blend:#ffffff,#6EB336", as_cmap=True)  # "#6EB336"

cmap_pept = sns.color_palette("blend:#ffffff,#d55e00", as_cmap=True)  # "#d55e00"


fig, axes = plt.subplots(2, 1, figsize=(8, 14), gridspec_kw={"hspace": 0.3})


# --- PB ---
sns.heatmap(
    pb_plot,
    cmap=cmap_prev,
    vmin=vmin,
    vmax=vmax,
    cbar=True,
    ax=axes[0],
    linewidths=0.3,
    linecolor="white",
)

axes[0].set_title("Prevotella bivia", fontsize=16, fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Clonotype")


# --- PL ---
sns.heatmap(
    pl_plot,
    cmap=cmap_pept,
    vmin=vmin,
    vmax=vmax,
    cbar=True,
    ax=axes[1],
    linewidths=0.3,
    linecolor="white",
)

axes[1].set_title("Peptoniphilus lacrimalis", fontsize=16, fontweight="bold")
axes[1].set_xlabel("Patient")
axes[1].set_ylabel("Clonotype")


plt.tight_layout()
plt.savefig(figures_dir / "6B_top_expanded_clonotypes.pdf", bbox_inches="tight")
plt.show()

### 4D. MAIT cell expansion

In [ ]:
# Load the discovery dataset
disc_data = pd.read_csv(discovery_processed / "0_final_data_parsed_annotated.csv")
disc_data = disc_data[
    [
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient",
        "cell_type",
        "chain",
        "full_tcr",
        "clonefreq_celltype",
    ]
].drop_duplicates()

# Compute clone frequency in stimulated data
final_data["total_clonecount_patient"] = final_data.groupby("patient")[
    "clonecount"
].transform("sum")
final_data["clone_frequency"] = (
    final_data["clonecount"] / final_data["total_clonecount_patient"]
)

# Identify alpha chains and MAIT cells based on their V- and J-genes
disc_data_alpha = disc_data[disc_data["chain"] == "TRA"]
stim_data_alpha = final_data[final_data["chain"] == "TRA"]

MAIT_alpha_bulk = disc_data_alpha[
    (disc_data_alpha["v_call"] == "TRAV1-2*01")
    & (
        disc_data_alpha["j_call"].isin(
            ["TRAJ33*01", "TRAJ12*01", "TRAJ20*01", "TRAJ30*01"]
        )
    )
]

MAIT_alpha_stim = stim_data_alpha[
    (stim_data_alpha["v_call"] == "TRAV1-2*01")
    & (
        stim_data_alpha["j_call"].isin(
            ["TRAJ33*01", "TRAJ12*01", "TRAJ20*01", "TRAJ30*01"]
        )
    )
]

# Split stimulated MAIT cells by condition
conditions = ["PL_1d", "PL_1w", "PB_1d", "PB_1w"]
MAIT_stim_dfs = {
    cond: MAIT_alpha_stim[MAIT_alpha_stim["condition"] == cond] for cond in conditions
}

# Function to compute range counts
bins = [0, 0.0001, 0.001, 0.01, 1]
labels = ["0% - 0.01%", "0.01% - 0.1%", "0.1% - 1%", ">1%"]


def compute_range_counts(df, value_col):
    df = df.copy()
    df["range"] = pd.cut(df[value_col], bins=bins, labels=labels, include_lowest=True)
    counts = df["range"].value_counts(normalize=True).sort_index() * 100
    return counts


# Compute range counts per condition
MAIT_counts_dict = {
    cond: compute_range_counts(df, "clone_frequency")
    for cond, df in MAIT_stim_dfs.items()
}
MAIT_counts_dict["MAIT_bulk"] = compute_range_counts(
    MAIT_alpha_bulk, "clonefreq_celltype"
)

# Combine
combined_counts = pd.DataFrame(MAIT_counts_dict).T.fillna(0)
row_order = ["MAIT_bulk", "PB_1d", "PB_1w", "PL_1d", "PL_1w"]
combined_counts = combined_counts.loc[row_order]

# Compute row-based percentages and unique junction counts
row_based_perc = {
    "MAIT_bulk": round(len(MAIT_alpha_bulk) / len(disc_data_alpha) * 100, 2),
    "PB_1d": round(
        len(MAIT_stim_dfs["PB_1d"])
        / len(stim_data_alpha[stim_data_alpha["condition"] == "PB_1d"])
        * 100,
        2,
    ),
    "PB_1w": round(
        len(MAIT_stim_dfs["PB_1w"])
        / len(stim_data_alpha[stim_data_alpha["condition"] == "PB_1w"])
        * 100,
        2,
    ),
    "PL_1d": round(
        len(MAIT_stim_dfs["PL_1d"])
        / len(stim_data_alpha[stim_data_alpha["condition"] == "PL_1d"])
        * 100,
        2,
    ),
    "PL_1w": round(
        len(MAIT_stim_dfs["PL_1w"])
        / len(stim_data_alpha[stim_data_alpha["condition"] == "PL_1w"])
        * 100,
        2,
    ),
}

unique_counts = {
    "MAIT_bulk": MAIT_alpha_bulk["junction_aa"].nunique(),
    "PB_1d": MAIT_stim_dfs["PB_1d"]["junction_aa"].nunique(),
    "PB_1w": MAIT_stim_dfs["PB_1w"]["junction_aa"].nunique(),
    "PL_1d": MAIT_stim_dfs["PL_1d"]["junction_aa"].nunique(),
    "PL_1w": MAIT_stim_dfs["PL_1w"]["junction_aa"].nunique(),
}

combined_counts

In [ ]:
# figure parameters and variables
x_labels = ["Bulk", "PB_1d", "PB_1w", "PL_1d", "PL_1w"]
fancy_colors = ["#feedde", "#fdbe85", "#fd8d3c", "#e6550d", "#a63603"]


#########################
### Figure generation ###
#########################
fig, ax = plt.subplots(figsize=(7, 12))

combined_counts.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=fancy_colors,
    edgecolor="white",
    width=0.9,
    legend=True,
)

ax.set_title("MAIT frequencies", fontsize=18, fontweight="bold", pad=60)
ax.set_xlabel("")
ax.set_ylabel("Percentage of repertoire", fontsize=16)
ax.set_xlim(-0.5, len(combined_counts) - 0.55)
ax.set_ylim(0, 100)
ax.set_yticks(np.arange(0, 101, 20))
ax.set_yticklabels([f"{y}%" for y in np.arange(0, 101, 20)])
ax.set_xticklabels(["Bulk data", "PB 1d", "PB 1w", "PL 1d", "PL 1w"], ha="center")
ax.tick_params(axis="x", rotation=0, labelsize=15)
ax.tick_params(axis="y", rotation=0, labelsize=13)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(
    frameon=True, loc="lower center", ncol=5, fontsize=12, bbox_to_anchor=(0.43, -0.11)
)


# Annotate bars with uniqeu TCRs and percentage
for i, idx in enumerate(combined_counts.index):
    total_height = combined_counts.loc[idx].sum()
    n_unique = unique_counts[idx]
    perc = row_based_perc[idx]
    ax.text(
        i,
        total_height + 1,
        f"n={n_unique}\n{perc:.2f}%",
        ha="center",
        va="bottom",
        fontsize=15,
    )

plt.tight_layout()
plt.savefig(figures_dir / "6C_MAIT_expansion.pdf", bbox_inches="tight")
plt.show()

### 4E. Per patient MAIT expansion

In [ ]:
# row_order = ["MAIT_bulk", "PB_1d", "PB_1w", "PL_1d", "PL_1w"]

MAIT_alpha_bulk["range"] = pd.cut(
    MAIT_alpha_bulk["clonefreq_celltype"], bins=bins, labels=labels, include_lowest=True
)
MAIT_alpha_stim["range"] = pd.cut(
    MAIT_alpha_stim["clone_frequency"], bins=bins, labels=labels, include_lowest=True
)

In [ ]:
patients = sorted(set(MAIT_alpha_stim["patient"].str.split("_").str[0]))

combined_list = []
for patient in patients:
    # Bulk
    MAIT_alpha_bulk["patient_stim"] = MAIT_alpha_bulk["patient"].str.split("_").str[0]
    bulk_row = MAIT_alpha_bulk[MAIT_alpha_bulk["patient_stim"] == patient]
    bulk_counts = (
        bulk_row["range"].value_counts(normalize=True).reindex(labels, fill_value=0)
        * 100
    )
    bulk_counts["patient"] = patient
    bulk_counts["condition"] = "Bulk"
    bulk_counts["n_unique"] = bulk_row["junction_aa"].nunique()
    combined_list.append(bulk_counts)
    # Stimulated
    for cond in ["PB_1d", "PB_1w", "PL_1d", "PL_1w"]:
        stim_row = MAIT_alpha_stim[
            (MAIT_alpha_stim["patient"].str.startswith(patient))
            & (MAIT_alpha_stim["condition"] == cond)
        ]
        stim_counts = (
            stim_row["range"].value_counts(normalize=True).reindex(labels, fill_value=0)
            * 100
        )
        stim_counts["patient"] = patient
        stim_counts["condition"] = cond
        stim_counts["n_unique"] = stim_row["junction_aa"].nunique()
        combined_list.append(stim_counts)

combined_df = pd.DataFrame(combined_list).fillna(0)
plot_df = combined_df.set_index(["patient", "condition"])[labels + ["n_unique"]]

In [ ]:
pb_conditions = ["Bulk", "PB_1d", "PB_1w"]
plot_MAIT_per_patient(
    plot_df,
    patients,
    labels,
    pb_conditions,
    "MAIT TCR: Bulk vs PB stimulations per patient",
    figures_dir,
)

pl_conditions = ["Bulk", "PL_1d", "PL_1w"]
plot_MAIT_per_patient(
    plot_df,
    patients,
    labels,
    pl_conditions,
    "MAIT TCR: Bulk vs PL stimulations per patient",
    figures_dir,
)

# 5. Negative shuffled networks and non-genus matches

### Create shuffled networks, load results and combine chain files

In [ ]:
os.makedirs(
    base_dir
    / "2_processed_data"
    / "11_negative_networks"
    / "1_processed_shuffled_networks",
    exist_ok=True,
)
all_files = os.listdir(
    base_dir / "2_processed_data" / "11_negative_networks" / "0_shuffled_PB_PL"
)

genera_to_use = ["Peptoniphilus", "Prevotella"]
chains = ["alpha", "beta"]
max_shuffles = 25


for genus in genera_to_use:
    # Loop over the files and select the Pept and Prev files
    genus_files = [
        f for f in all_files if genus in f and any(chain in f for chain in chains)
    ]

    iter_dict = {}
    for f in genus_files:
        m = re.search(r"iteration(\d+)\.csv", f)
        if m:
            iter_num = int(m.group(1))
            iter_dict.setdefault(iter_num, []).append(f)

    sorted_iters = sorted(iter_dict.keys())[:max_shuffles]

    # Process each iteration
    for iter_num in sorted_iters:
        files_to_load = iter_dict[iter_num]
        dfs = []

        for file in files_to_load:
            file_path = os.path.join(
                base_dir
                / "2_processed_data"
                / "11_negative_networks"
                / "0_shuffled_PB_PL",
                file,
            )
            df = pd.read_csv(file_path, index_col=0)
            df = df[(df["convergence"] > 2) & (df["pvalue"] <= 0.05)]
            df["chain"] = df["v_call"].str.extract(r"(TR[A|B])")
            df["genus"] = genus

            # Count patients per TCR
            tcr_patient_counts = (
                df.groupby(["junction_aa", "v_call", "j_call"])["patient_id"]
                .nunique()
                .reset_index()
                .rename(columns={"patient_id": "patient_count"})
            )
            df = df.merge(
                tcr_patient_counts, on=["junction_aa", "v_call", "j_call"], how="left"
            )

            df = df[
                [
                    "junction_aa",
                    "v_call",
                    "j_call",
                    "chain",
                    "genus",
                    "convergence",
                    "patient_count",
                ]
            ].drop_duplicates()

            dfs.append(df)

        if dfs:
            # Combine alpha + beta
            combined_df = pd.concat(dfs, ignore_index=True)

            all_nodes_shuffled = pd.concat(
                [combined_df["junction_aa"], combined_df["genus"]]
            ).unique()
            node_map_shuffled = {
                name: i + 1 for i, name in enumerate(all_nodes_shuffled)
            }
            combined_df["node_id_var1"] = combined_df["junction_aa"].map(
                node_map_shuffled
            )
            combined_df["node_id_var2"] = combined_df["genus"].map(node_map_shuffled)

            combined_df.to_csv(
                os.path.join(
                    base_dir
                    / "2_processed_data"
                    / "11_negative_networks"
                    / "1_processed_shuffled_networks",
                    f"{genus}_iteration{iter_num}_combined.csv",
                )
            )
            print(f"Saved combined shuffled network {genus} {iter_num}")

### Load processed shuffled networks and actual interactions

In [ ]:
shuffled_networks_by_genus = defaultdict(list)

shuffled_files = sorted(
    (
        base_dir
        / "2_processed_data"
        / "11_negative_networks"
        / "1_processed_shuffled_networks"
    ).glob("*.csv")
)


for f in shuffled_files:
    df = pd.read_csv(f, index_col=0)

    filename = f.name

    if "Prevotella" in filename:
        shuffled_networks_by_genus["Prevotella"].append(df)
    elif "Peptoniphilus" in filename:
        shuffled_networks_by_genus["Peptoniphilus"].append(df)

final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv",
    index_col=0,
)

conv_erc = final_conv[
    (final_conv["convergence"] > 2)
    & (final_conv["pvalue"] <= 0.05)
    & (final_conv["dataset"] == "Discovery")
]
conv_erc["chain"] = conv_erc["v_call"].str.extract(r"(TR[A|B])")

# Count patients per TCR
tcr_patient_counts = (
    conv_erc.groupby(["junction_aa", "v_call", "j_call"])["patient_id"]
    .nunique()
    .reset_index()
    .rename(columns={"patient_id": "patient_count"})
)
conv_erc = conv_erc.merge(
    tcr_patient_counts, on=["junction_aa", "v_call", "j_call"], how="left"
)

# Keep public TCRs: present in ≥5 patients
network_df = conv_erc[
    [
        "junction_aa",
        "v_call",
        "j_call",
        "chain",
        "genus",
        "convergence",
        "patient_count",
    ]
].drop_duplicates()

network_df["full_tcr"] = (
    network_df["v_call"] + "_" + network_df["junction_aa"] + "_" + network_df["j_call"]
)


sorted_genera = [
    "Peptoniphilus",
    "Lactobacillus",
    "Desulfovibrio",
    "Porphyromonas",
    "Erysipelotrichaceae",
    "Actinomyces",
    "Dialister",
    "Phascolarctobacterium",
    "Oxalobacter",
    "Paraprevotella",
    "Akkermansia",
    "Parasutterella",
    "Coprococcus",
    "Odoribacter",
    "Coprobacter",
    "Bilophila",
    "Bacteroidales",
    "Sutterella",
    "Prevotella",
    "Streptococcus",
    "Ruminococcaceae",
]  # The bacterial genera that have shared clusters across cohorts

network_df = network_df[network_df["genus"].isin(sorted_genera)]

### Load validation data

### Compute exact TCR overlap between validation data and original versus shuffled networks

In [ ]:
# Configuration

use_percent = True  # True = percent_overlap, False = absolute n_overlap_with_lab
metric_col = "percent_overlap" if use_percent else "n_overlap_with_lab"
p_value_type = "p_zscore"  # "p_empirical" or "p_zscore"
ylabel = "Percent Overlap (%)" if use_percent else "Overlap (# TCRs)"


# SHUFFLED NETWORK ANALYSIS

# Real overlaps
real_results = [
    compute_overlap_simple(network_df, PL_data, "Peptoniphilus", "real"),
    compute_overlap_simple(network_df, PB_data, "Prevotella", "real"),
]
overlap_real_df = pd.concat(real_results, ignore_index=True)

# Shuffled overlaps
shuffled_results = []
for genus in ["Peptoniphilus", "Prevotella"]:
    lab_df = PL_data if genus == "Peptoniphilus" else PB_data
    for i, shuf_df in enumerate(shuffled_networks_by_genus[genus]):
        shuffled_results.append(
            compute_overlap_simple(shuf_df, lab_df, genus, "shuffled", i)
        )
overlap_shuffled_df = pd.concat(shuffled_results, ignore_index=True)

overview_df = pd.concat([overlap_real_df, overlap_shuffled_df], ignore_index=True)
overview_df.to_csv(tables_dir / "3_shuffled_overview.csv", index=False)

# Compute stats using chosen metric
results = []
for genus in ["Peptoniphilus", "Prevotella"]:
    for chain in ["TRA", "TRB"]:
        real_val = overview_df[
            (overview_df["genus"] == genus)
            & (overview_df["chain"] == chain)
            & (overview_df["network_type"] == "real")
        ][metric_col].values[0]
        shuf_vals = overview_df[
            (overview_df["genus"] == genus)
            & (overview_df["chain"] == chain)
            & (overview_df["network_type"] == "shuffled")
        ][metric_col].values
        z, p_z, p_emp = compute_stats(real_val, shuf_vals)
        results.append(
            {
                "analysis": "shuffled_network",
                "genus": genus,
                "chain": chain,
                "real_overlap": real_val,
                "shuffled_mean": np.mean(shuf_vals),
                "shuffled_std": np.std(shuf_vals, ddof=1),
                "z_score": z,
                "p_zscore": p_z,
                "p_empirical": p_emp,
            }
        )

shuffled_stats_df = pd.DataFrame(results)

#######################################
# 5. CONVERGENCE OVERLAP
#######################################

PL_overlap, PL_stats = compute_overlap_conv_with_stats(
    network_df,
    PL_data,
    "Peptoniphilus",
    remove_small=True,
    min_tcrs=3000,
    metric_col=metric_col,
)
PB_overlap, PB_stats = compute_overlap_conv_with_stats(
    network_df,
    PB_data,
    "Prevotella",
    remove_small=True,
    min_tcrs=3000,
    metric_col=metric_col,
)

PL_stats["analysis"] = "convergence"
PB_stats["analysis"] = "convergence"
convergence_stats_df = pd.concat([PL_stats, PB_stats], ignore_index=True)
PL_overlap.to_csv(tables_dir / "3_PL_overview_overlap.csv", index=False)
PB_overlap.to_csv(tables_dir / "3_PB_overview_overlap.csv", index=False)


# Combine everything into one dataframe for plotting
combined_stats_df = pd.concat(
    [shuffled_stats_df, convergence_stats_df], ignore_index=True
)
combined_stats_df = combined_stats_df.sort_values(
    ["analysis", "genus", "chain"]
).reset_index(drop=True)

In [ ]:
###########################################
# 7. PLOTTING. Z-score / Empirical p-values
###########################################

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
chain_order = ["TRA", "TRB"]

ylabel = "Percent Overlap (%)" if use_percent else "Overlap (# TCRs)"

# --- A+B: shuffled networks ---
for i, genus in enumerate(["Prevotella", "Peptoniphilus"]):
    ax = axes[0, i]
    data = overview_df[overview_df["genus"] == genus]

    # Violin outline only
    sns.violinplot(
        data=data[data["network_type"] == "shuffled"],
        x="chain",
        y=metric_col,
        color="grey",
        fill=None,
        inner=None,
        linewidth=1.2,
        ax=ax,
    )

    # Grey points for shuffled distribution
    sns.stripplot(
        data=data[data["network_type"] == "shuffled"],
        x="chain",
        y=metric_col,
        color="gray",
        jitter=True,
        size=5,
        ax=ax,
    )

    # Red points for real network
    sns.stripplot(
        data=data[data["network_type"] == "real"],
        x="chain",
        y=metric_col,
        color="red",
        size=9,
        ax=ax,
    )

    # p-value annotation
    ymax = data[metric_col].max(skipna=True)
    for j, chain in enumerate(chain_order):
        p_val = shuffled_stats_df[
            (shuffled_stats_df["genus"] == genus)
            & (shuffled_stats_df["chain"] == chain)
        ][p_value_type].values[
            0
        ]  # p_zscore	p_empirical
        ax.text(j, ymax * 1.1, f"p={p_val:.3f}", ha="center")

    ax.set_title(genus)
    sns.despine(ax=ax)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)

# --- C+D: convergence overlap ---
for i, (genus, stats) in enumerate(
    [("Prevotella", PB_stats), ("Peptoniphilus", PL_stats)]
):
    ax = axes[1, i]
    df = PB_overlap if genus == "Prevotella" else PL_overlap

    # Violin outline only
    sns.violinplot(
        data=df,
        x="chain",
        y=metric_col,
        color="grey",
        fill=None,
        inner=None,
        linewidth=1.2,
        ax=ax,
    )

    # Black points for all values
    sns.stripplot(
        data=df,
        x="chain",
        y=metric_col,
        color="gray",
        jitter=True,
        size=5,
        ax=ax,
    )

    # Red dot for the target genus
    ymax = df[metric_col].max(skipna=True)
    for j, chain in enumerate(chain_order):
        val = stats[stats["chain"] == chain]["value"].values[0]
        ax.scatter(j, val, color="red", s=120, zorder=5)
        p_val = stats[stats["chain"] == chain][p_value_type].values[
            0
        ]  # p_zscore	p_empirical
        ax.text(j, ymax * 1.1, f"p={p_val:.3f}", ha="center")

    ax.set_title(genus)
    sns.despine(ax=ax)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)

plt.tight_layout()
plt.savefig(
    figures_dir / "7_overlap_violin.pdf",
    bbox_inches="tight",
)
plt.show()

# Experiment 2: Treg and Tconv stimulation with Peptoniphilus lacrimalis

## 1. Read in the PL experimental data

### 1A. Process raw files

In [ ]:
# Get the sequencing data from the second stimulation experiment
ID_mapping = pd.read_excel(raw_exp2_data / "2_PL_validation_IDs.xlsx")
ID_mapping_dict = dict(zip(ID_mapping["run_id"], ID_mapping["sample_id"]))

data_dir = raw_exp2_data / "0_PL_TCRs"

patient_data = []

for file_name in os.listdir(data_dir):
    if file_name.endswith(".tsv"):
        df = pd.read_csv(os.path.join(data_dir, file_name), sep="\t")
        sample_name = file_name.split("_")[4]
        df["filename"] = sample_name
        patient_data.append(df)

result_df = pd.concat(patient_data, ignore_index=True)

result_df["run_id"] = result_df["filename"].map(ID_mapping_dict)
result_df[["Sample", "Cell_type"]] = result_df["run_id"].str.split("_", expand=True)

### 1B. TCR parsing

In [ ]:
# Format the data
Full_parsed = result_df[
    [
        "cloneId",
        "readCount",
        "readFraction",
        "allVHitsWithScore",
        "allJHitsWithScore",
        "nSeqCDR3",
        "aaSeqCDR3",
        "filename",
        "run_id",
        "Sample",
        "Cell_type",
    ]
]

Full_parsed = Full_parsed.rename(
    columns={
        "nSeqCDR3": "junction",
        "allVHitsWithScore": "v_call",
        "allJHitsWithScore": "j_call",
        "aaSeqCDR3": "junction_aa",
        "Sample": "patient_id",
        "Cell_type": "cell_type",
    }
)

# Parse the TCR data
Full_parsed = parse_mixcr(Full_parsed, rename=False)

# Add additional metadata to the dataframe
Full_parsed["chain"] = Full_parsed["v_call"].str.extract(r"(TR[A|B])")

Full_parsed["clonecount"] = Full_parsed.groupby(
    ["junction", "v_call", "j_call", "junction_aa", "patient_id", "cell_type", "chain"]
)["readCount"].transform("sum")

Full_parsed = Full_parsed.drop(
    columns=["cloneId", "readCount", "readFraction", "filename"]
).drop_duplicates()

# Count the number of unique TCRs per patient and chain
count_table = (
    Full_parsed.groupby(["run_id", "chain"])["junction_aa"].nunique().reset_index()
)

count_table.columns = ["run_id", "chain", "unique_tcr_count"]

## 2. Plotting data overview

### 2A. Unique TCRs per patient, cell type and chain

In [ ]:
# Get the number of unique TCRs per group
grouped = (
    Full_parsed.groupby(["patient_id", "cell_type", "chain"])["junction_aa"]
    .nunique()
    .reset_index()
    .pivot_table(
        index=["patient_id", "cell_type"],
        columns="chain",
        values="junction_aa",
        fill_value=0,
    )
    .reset_index()
)

grouped.columns.name = None
patient_ids = grouped["patient_id"].unique()
chains = ["TRA", "TRB"]


# Color palette
colors = {
    "Tconv_TRA": "#003049",
    "Tconv_TRB": "#669bbc",
    "Treg_TRA": "#6C0F32",
    "Treg_TRB": "#FF858D",
}

#########################
### Figure generation ###
#########################

fig, ax = plt.subplots(figsize=(10, 8))
bar_width = 0.35
x = range(len(patient_ids))

tconv_vals = grouped[grouped["cell_type"] == "Tconv"]
treg_vals = grouped[grouped["cell_type"] == "Treg"]


def get_vals(df, chain):
    return [
        (
            df[df["patient_id"] == p][chain].values[0]
            if p in df["patient_id"].values
            else 0
        )
        for p in patient_ids
    ]


# Plot Tconv bars
bottom = [0] * len(patient_ids)
for chain in chains:
    heights = get_vals(tconv_vals, chain)
    ax.bar(
        [p - bar_width / 2 for p in x],
        heights,
        bar_width,
        label=f"Tconv - {chain}",
        bottom=bottom,
        color=colors[f"Tconv_{chain}"],
    )
    bottom = [b + h for b, h in zip(bottom, heights)]

# Plot Treg bars
bottom = [0] * len(patient_ids)
for chain in chains:
    heights = get_vals(treg_vals, chain)
    ax.bar(
        [p + bar_width / 2 for p in x],
        heights,
        bar_width,
        label=f"Treg - {chain}",
        bottom=bottom,
        color=colors[f"Treg_{chain}"],
    )
    bottom = [b + h for b, h in zip(bottom, heights)]

ax.set_xticks(x)
ax.set_xticklabels(patient_ids, rotation=0, ha="center")
ax.tick_params(axis="x", labelsize=16)
ax.tick_params(axis="y", labelsize=16)
ax.set_ylabel("Unique TCR Count", size=18)
ax.set_title("")
ax.legend(title="", fontsize=15)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(figures_dir / "S11A_validation2_counts.pdf", bbox_inches="tight")
plt.show()

### 2B. Expansion plot

In [ ]:
# Get expansion ranges per cell type
Full_treg = Full_parsed[Full_parsed["cell_type"] == "Treg"]
Full_tconv = Full_parsed[Full_parsed["cell_type"] == "Tconv"]

treg_exp = get_expansion(Full_treg, "patient_id", "patient_id", type="frequency")
tconv_exp = get_expansion(Full_tconv, "patient_id", "patient_id", type="frequency")

#########################
### Figure generation ###
#########################

colors = ["#feedde", "#fdbe85", "#fd8d3c", "#e6550d", "#a63603"]
fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)

for ax, dfs in zip(axes, [treg_exp, tconv_exp]):
    df_concat = dfs.copy()

    df_concat.plot(
        kind="bar",
        stacked=True,
        color=colors[: len(df_concat.columns)],
        width=0.9,
        ax=ax,
        legend=False,
    )

    ax.set_xticks(range(len(df_concat)))
    ax.set_xticklabels(df_concat.index, rotation=0, ha="center", fontsize=12)
    ax.set_title("")
    ax.set_xlabel("", fontsize=14)
    ax.set_ylabel("Frequency", fontsize=14)
    ax.set_ylim(0, 1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title("Treg" if dfs is treg_exp else "Tconv", fontsize=16)


handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(labels),
    fontsize=14,
    frameon=True,
    bbox_to_anchor=(0.5, -0.05),
)

plt.tight_layout()
plt.savefig(figures_dir / "S11B_validation2_expansion.pdf", bbox_inches="tight")
plt.show()

## 3. Cell type enrichment in convergent TCRs

In [ ]:
# Load all convergent TCRs
final_conv = pd.read_csv(discovery_conv / "0_combined_final_sign_interactions.csv")
final_conv = final_conv[final_conv["dataset"] == "Discovery"]

Full_parsed.rename(columns={"patient_id": "patient"}, inplace=True)

# Merge convergence info into full dataset
full_tcr_info = pd.merge(
    Full_parsed,
    final_conv[
        [
            "junction",
            "junction_aa",
            "v_call",
            "j_call",
            "patient_id",
            "statistic",
            "pvalue",
            "convergence",
            "genus",
        ]
    ],
    on=["junction_aa"],
    how="left",
)

full_tcr_info = full_tcr_info[
    [
        "junction_x",
        "junction_aa",
        "v_call_x",
        "j_call_x",
        "patient",
        "run_id",
        "patient_id",
        "cell_type",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
    ]
]

full_tcr_info["convergent"] = np.where(
    (full_tcr_info["convergence"] > 2) & (full_tcr_info["pvalue"] < 0.05), "Yes", "No"
)

# Remove duplicates per TCR/genus/convergence combination
full_tcr_info = full_tcr_info.drop_duplicates(
    subset=[
        "junction_aa",
        "patient",
        "cell_type",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
        "convergent",
    ]
)

# Baseline repertoire (all TCRs unique by junction_aa/patient/cell_type)
before_df = full_tcr_info.drop_duplicates(
    subset=["junction_aa", "patient", "cell_type"]
).drop(columns=["statistic", "pvalue", "convergence", "genus", "convergent"])


# Get overview of Treg versus Tconv distribution in patients and convergent TCRs
res_df_all = build_overview(full_tcr_info, before_df, sorted_genera, "validation2")

# Pivot for heatmap
heatmap = res_df_all.pivot_table(
    index=["dataset", "cell_type"], columns="genus", values="Log2FC", aggfunc="mean"
)
heatmap = heatmap.reindex(columns=sorted_genera).fillna(0)

# split run rows and aggregated rows
run_rows = [ct for d, ct in heatmap.index if not ct.endswith("_agg")]
agg_rows = [ct for d, ct in heatmap.index if ct.endswith("_agg")]
run_rows_sorted = sorted(run_rows, key=lambda x: (x.split("_")[1], x.split("_")[0]))


final_index = [("validation2", ct) for ct in run_rows_sorted + agg_rows]
heatmap_disp = heatmap.loc[final_index]
heatmap_disp.index = [f"   | {ct}" for d, ct in heatmap_disp.index]


#########################
### Figure generation ###
#########################
g = sns.clustermap(
    heatmap_disp,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    row_cluster=False,
    col_cluster=True,
    figsize=(14, 10),
    dendrogram_ratio=(0.000001, 0.0000001),
    cbar_kws={"label": "Log2FC"},
    cbar_pos=(1.0, 0.3, 0.03, 0.6),
    xticklabels=True,
    yticklabels=True,
)

g.ax_heatmap.hlines(
    len(run_rows_sorted), *g.ax_heatmap.get_xlim(), colors="white", linewidth=6
)

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, ha="center")
plt.title("", pad=20)
plt.savefig(
    figures_dir / "S11C_validation2_subtype_heatmap_21.pdf", bbox_inches="tight"
)
plt.show()

## 4. Overlap with the original discovery cohort

In [ ]:
# Original discovery dataset
disc_data = pd.read_csv(discovery_processed / "0_final_data_parsed_annotated.csv")
disc_data = disc_data[
    ["junction_aa", "v_call", "j_call", "patient", "cell_type", "chain", "full_tcr"]
]

# Convergent TCRs in the discovery dataset
final_conv = pd.read_csv(discovery_conv / "0_combined_final_sign_interactions.csv")
final_conv = final_conv[
    (final_conv["dataset"] == "Discovery")
    & (final_conv["convergence"] > 2)
    & (final_conv["pvalue"] < 0.05)
].rename(columns={"patient_id": "patient"})
final_conv["chain"] = final_conv["v_call"].str.extract(r"(TR[A|B])")

# Stimulated TCRs
final_data = Full_parsed[
    [
        "clonecount",
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient",
        "run_id",
        "chain",
        "cell_type",
    ]
].rename(columns={"patient": "patient_id", "run_id": "patient"})

final_data["full_tcr"] = (
    final_data["v_call"] + "_" + final_data["junction_aa"] + "_" + final_data["j_call"]
)

# Split on cell type
Treg_data = final_data[final_data["cell_type"] == "Treg"]
Tconv_data = final_data[final_data["cell_type"] == "Tconv"]

# Calculate overlap numbers with disovery dataset
Treg_summary_full = summarize_overlap_conv(Treg_data, disc_data, final_conv, "Treg")
Tconv_summary_full = summarize_overlap_conv(Tconv_data, disc_data, final_conv, "Tconv")

# Combine
data_summary_full = pd.concat(
    [Treg_summary_full, Tconv_summary_full], ignore_index=True
)
data_summary_full = data_summary_full.set_index(["patient_id", "cell_type"])

data_summary_full.to_csv(exp_processed_data / "2_PL" / "1_overlap_overview_numbers.csv")

# Get overlapping TCR sequences for the regulatory T cells
overlap_conv_treg = pd.merge(Treg_data, final_conv, on="junction_aa", how="inner")
overlap_conv_treg = overlap_conv_treg[
    [
        "junction_aa",
        "patient_x",
        "cell_type_x",
        "clonecount",
        "patient_y",
        "genus",
        "pvalue",
        "convergence",
    ]
].drop_duplicates()
overlap_conv_treg.to_csv(exp_processed_data / "2_PL" / "2_all_conv_overlap_treg.csv")

# Get overlapping TCR sequences for the conventional T cells
overlap_conv_tconv = pd.merge(Tconv_data, final_conv, on="junction_aa", how="inner")
overlap_conv_tconv = overlap_conv_tconv[
    [
        "junction_aa",
        "patient_x",
        "cell_type_x",
        "clonecount",
        "patient_y",
        "genus",
        "pvalue",
        "convergence",
    ]
].drop_duplicates()
overlap_conv_tconv.to_csv(exp_processed_data / "2_PL" / "2_all_conv_overlap_tconv.csv")

## 5. Overlap with the convergent TCRs

In [ ]:
list_of_dfs = []

for genus in sorted_genera:
    df = pd.read_csv(
        base_dir
        / "2_processed_data"
        / "7_cluster_convergence"
        / f"0_clustered_{genus}.csv"
    )

    list_of_dfs.append(df)
combined_df = pd.concat(list_of_dfs, ignore_index=True)

combined_df["full_tcr"] = (
    combined_df["v_call"]
    + "_"
    + combined_df["junction_aa"]
    + "_"
    + combined_df["j_call"]
)

combined_df_erc = combined_df[combined_df["dataset"] == "Discovery"]

# compute_overlap_conv already defined
df1, df2 = compute_overlap_clusters_conv(
    combined_df_erc,
    Treg_data,
    "cell_type",
    "Peptoniphilus",
    remove_small=True,
    min_tcrs=3000,
)
df3, df4 = compute_overlap_clusters_conv(
    combined_df_erc,
    Tconv_data,
    "cell_type",
    "Peptoniphilus",
    remove_small=True,
    min_tcrs=3000,
)

#########################
### Figure generation ###
#########################

plot_relative_overlap(
    panels={
        "Treg": (df1, df2, "Peptoniphilus"),
        "Tconv": (df3, df4, "Peptoniphilus"),
    },
    savename=figures_dir / "S11D_validation2_violin_overlap_selected_genera_th3000.pdf",
)